# Real-time Banking Fraud Detection — Live Dashboard

This notebook is the **monitoring dashboard** for Bank X's fraud-detection platform.

It reads the Parquet files continuously produced by the **Spark Streaming processor**
(shared `/data/output` volume) and refreshes automatically.

It displays, for the **last 20 users** seen in transactions:

- Average amounts sent / received over the **3h / 7d / 3w / 3m** sliding windows
- Transaction counts and distinct network size (senders / receivers) per window
- Lifetime ("since account creation") totals and periodic averages
- Activity over the **last 10 seconds**
- Charts and **red highlighting** for anomalous spending spikes

**How to use:** run all cells top to bottom. The last cell starts the auto-refresh
loop — interrupt the kernel (■ stop button) to stop it.


In [13]:
# --- Configuration and imports --------------------------------------------
import os
import time
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

# The Spark processor writes its results here (shared Docker volume).
OUTPUT_DIR   = os.environ.get("DATA_DIR", "/workspace/data") + "/output"
USER_METRICS = OUTPUT_DIR + "/user_metrics"
RECENT_TX    = OUTPUT_DIR + "/recent_tx"

REFRESH_SECONDS = 5     # auto-refresh interval
ITERATIONS      = 120   # number of refreshes (120 x 5s = 10 minutes)
N_USERS         = 20    # show the last N users that appeared
ACTIVITY_WINDOW = 10    # "last 10 seconds of activity"

print("Dashboard configured. Reading Spark output from:", OUTPUT_DIR)


Dashboard configured. Reading Spark output from: /workspace/data/output


In [14]:
# --- Helper functions ------------------------------------------------------
def load_parquet(path):
    # Read a Parquet directory written by Spark.
    # The processor overwrites these directories every few seconds, so a read
    # may occasionally land mid-write -> we simply retry a couple of times.
    for _ in range(3):
        try:
            return pd.read_parquet(path)
        except Exception:
            time.sleep(0.5)
    return None


def get_recent_users(recent, n):
    # Return the last `n` distinct users that appeared in transactions,
    # counting both senders and receivers, most-recent first.
    events = pd.concat([
        recent[["ts", "send_id"]].rename(columns={"send_id": "user"}),
        recent[["ts", "receive_id"]].rename(columns={"receive_id": "user"}),
    ])
    events = events.sort_values("ts", ascending=False)
    return list(dict.fromkeys(events["user"]))[:n]


def highlight_anomalies(row):
    # pandas Styler callback: paint a row red when it is flagged anomalous.
    spike = bool(row.get("anomaly", False))
    return ["background-color: #ffcccc" if spike else "" for _ in row]


In [15]:
# --- The dashboard renderer ------------------------------------------------
def render():
    recent  = load_parquet(RECENT_TX)
    print("loaded the parquet")
    metrics = load_parquet(USER_METRICS)
    
    clear_output(wait=True)
    print("=" * 78)
    print("   REAL-TIME BANKING FRAUD DETECTION  -  Bank X monitoring committee")
    print("   refreshed at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 78)

    if metrics is None or len(metrics) == 0:
        print("\nWaiting for the Spark processor's first batch...")
        print("Open a Jupyter terminal and run:  python processor/processor.py")
        return
    # If recent_tx is briefly empty (processor still draining backlog), we keep
    # showing the per-user metrics table instead of blanking the dashboard.
    no_recent = recent is None or len(recent) == 0

    if not no_recent:
        recent["ts"] = pd.to_datetime(recent["ts"])
        now = recent["ts"].max()
        last_window = recent[recent["ts"] >= now - timedelta(seconds=ACTIVITY_WINDOW)]
        tps = len(last_window) / ACTIVITY_WINDOW
        print(f"\nActivity in the last {ACTIVITY_WINDOW}s : {len(last_window)} "
              f"transactions   (~{tps:.1f} tx/s)")
        print(f"Distinct users currently tracked by Spark : {len(metrics)}")
        users = get_recent_users(recent, N_USERS)
        view = metrics[metrics["user"].isin(users)].copy()
        view["__order"] = view["user"].apply(lambda u: users.index(u))
        view = view.sort_values("__order").drop(columns="__order")
    else:
        print(f"\n[ recent_tx is briefly empty - processor catching up to live data ]")
        print(f"Distinct users currently tracked by Spark : {len(metrics)}")
        # Show the most active users by 3h send count instead.
        view = metrics.dropna(subset=["count_sent_3h"]).sort_values(
            "count_sent_3h", ascending=False).head(N_USERS).copy()
        last_window = None

    # ---- Fraud detection: three rules combined -----------------------------
    # Rule A - AMOUNT SPIKE: recent (3h) avg amount > 3x lifetime avg,
    #   with at least 3 recent transactions
    ratio = (view["avg_amount_sent_3h"].fillna(0) /
             view["avg_amount_sent_life"].replace(0, np.nan)).fillna(0)
    flag_A = (ratio > 3) & (view["count_sent_3h"].fillna(0) >= 3)
    # Rule B - NETWORK FAN-OUT: many distinct receivers in 3h, abnormally
    #   compared to the user's lifetime network size
    # Rule B - NETWORK FAN-OUT: many distinct receivers in 3h AND
    #   non-trivial amounts (suppresses tiny-amount legitimate users)
    flag_B = ((view["distinct_sent_3h"].fillna(0) >= 15) &
              (view["avg_amount_sent_3h"].fillna(0) >= 20))
    # Rule C - ACTIVITY BURST: many transactions in 3h AND non-trivial amounts
    flag_C = ((view["count_sent_3h"].fillna(0) >= 15) &
              (view["avg_amount_sent_3h"].fillna(0) >= 20))
    view["flag_amount"]  = flag_A
    view["flag_network"] = flag_B
    view["flag_burst"]   = flag_C
    view["anomaly"] = flag_A | flag_B | flag_C
    view["fraud_reasons"] = (
        flag_A.map({True: "AMOUNT ", False: ""}) +
        flag_B.map({True: "NETWORK ", False: ""}) +
        flag_C.map({True: "BURST", False: ""})
    ).str.strip()

    # ---- Top of dashboard: FRAUD ALERTS panel ------------------------------
    n_anom = int(view["anomaly"].sum())
    if n_anom:
        print(f"\n*** FRAUD ALERTS  ({n_anom} user(s) flagged) ***")
        alerts = (view[view["anomaly"]]
                  [["user", "bank", "fraud_reasons",
                    "count_sent_3h", "avg_amount_sent_3h",
                    "distinct_sent_3h", "avg_amount_sent_life"]]
                  .fillna(0).round(2)
                  .rename(columns={
                      "avg_amount_sent_3h": "avg_amount_3h",
                      "avg_amount_sent_life": "avg_amount_lifetime",
                      "count_sent_3h": "tx_count_3h",
                      "distinct_sent_3h": "distinct_recv_3h",
                      "fraud_reasons": "triggered_rules",
                  }))
        display(alerts.style.set_properties(**{"background-color": "#ffd6d6"})
                .hide(axis="index"))
    else:
        print("\n[ No fraud alerts among the displayed users ]")

    # ---- Table 1 : average amounts per window -----------------------------
    print(f"\n--- AVERAGE AMOUNTS  (sent / received)  -  last {N_USERS} users ---")
    cols = ["user", "bank"]
    for w in ["3h", "7d", "3w", "3m", "life"]:
        cols += [f"avg_amount_sent_{w}", f"avg_amount_recv_{w}"]
    t1 = view[cols + ["anomaly"]].fillna(0).round(2)
    display(t1.style.apply(highlight_anomalies, axis=1).hide(axis="index"))

    # ---- Table 2 : counts and distinct network per window -----------------
    print(f"\n--- TRANSACTION COUNTS & DISTINCT NETWORK  -  last {N_USERS} users ---")
    cols2 = ["user", "bank"]
    for w in ["3h", "7d", "3w", "3m", "life"]:
        cols2 += [f"count_sent_{w}", f"count_recv_{w}",
                  f"distinct_sent_{w}", f"distinct_recv_{w}"]
    t2 = view[cols2 + ["anomaly"]].fillna(0)
    display(t2.style.apply(highlight_anomalies, axis=1).hide(axis="index"))

    # ---- Table 3 : since account creation ---------------------------------
    print(f"\n--- SINCE ACCOUNT CREATION (lifetime)  -  last {N_USERS} users ---")
    cols3 = ["user", "bank", "total_sent", "total_recv",
             "count_sent_life", "count_recv_life",
             "distinct_sent_life", "distinct_recv_life",
             "avg_hourly_sent", "avg_daily_sent",
             "avg_weekly_sent", "avg_monthly_sent"]
    t3 = view[cols3].fillna(0).round(2)
    display(t3.style.hide(axis="index"))

    # ---- Charts -----------------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    if not no_recent:
        per_sec = (recent[recent["ts"] >= now - timedelta(seconds=60)]
                   .set_index("ts").resample("1s").size())
        ax[0].plot(per_sec.index, per_sec.values, marker="o", color="steelblue")
        ax[0].set_title("Transactions per second (last 60s)")
    else:
        ax[0].text(0.5, 0.5, "recent_tx not yet available\n(processor catching up)",
                   ha="center", va="center", transform=ax[0].transAxes,
                   color="gray")
        ax[0].set_title("Transactions per second (last 60s)")
    ax[0].set_xlabel("time")
    ax[0].set_ylabel("tx/s")
    ax[0].tick_params(axis="x", rotation=45)
    ax[0].grid(alpha=0.3)

    top = view.sort_values("count_sent_3h", ascending=False).head(10)
    colors = ["red" if a else "steelblue" for a in top["anomaly"]]
    ax[1].barh(top["user"].astype(str), top["count_sent_3h"].fillna(0), color=colors)
    ax[1].set_title("Top displayed users by tx sent (3h window)")
    ax[1].set_xlabel("transactions sent")
    ax[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    n_anom = int(view["anomaly"].sum())
    if n_anom:
        print(f"\n[!] {n_anom} user(s) flagged by fraud rules (red rows). "
              f"Rules triggered: AMOUNT spike | NETWORK fan-out | activity BURST.")
    else:
        print("\nNo anomalies among the displayed users.")


## Live dashboard

Run the cell below to start the auto-refreshing dashboard.

- It refreshes every **5 seconds** and reads the latest Spark output each time.
- Rows highlighted in **red** are users whose short-term (3h) average amount is
  more than 3x their lifetime average — a simple fraud heuristic.
- Stop it any time with the kernel **interrupt / stop** button.


In [ ]:
# --- Start the live auto-refreshing dashboard ------------------------------
# Interrupt the kernel (the square stop button) to stop the loop.
try:
    for i in range(ITERATIONS):
        render()
        print(f"\n(refresh {i + 1}/{ITERATIONS}  -  next update in "
              f"{REFRESH_SECONDS}s  -  interrupt the kernel to stop)")
        time.sleep(REFRESH_SECONDS)
    print("Reached the configured number of refreshes. Re-run this cell to continue.")
except KeyboardInterrupt:
    print("Dashboard stopped by user.")


   REAL-TIME BANKING FRAUD DETECTION  -  Bank X monitoring committee
   refreshed at 2026-05-21 09:52:20

Waiting for the Spark processor's first batch...
Open a Jupyter terminal and run:  python processor/processor.py

(refresh 109/120  -  next update in 5s  -  interrupt the kernel to stop)
